## LSTM for Coffee Price Forecasting

# 1. Environment Setup

In [5]:
from IPython.display import display, HTML

display(HTML("<style>.container {width: 100% !important;}</style>"))

## 1.1. Library Imports

In [6]:
import torch
import subprocess
import sys

import torch
import torchvision

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

## 1.2. GPU Check

In [7]:
if torch.cuda.is_available():
    print("__CUDNN VERSION:", torch.backends.cudnn.version())
    print("Device Name:", torch.cuda.get_device_name(0))
    device = 'cuda'
else:
    print("CUDA is not available.")
    device = 'cpu'

print('Device:', device)

__CUDNN VERSION: 91900
Device Name: NVIDIA GeForce RTX 5070
Device: cuda


## 1.3. Export requirements

In [8]:
def export_requirements():
    try:
        result = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                                capture_output=True,
                                text=True,
                                check=True)
        with open('requirements.txt', 'w') as f:
            f.write(result.stdout)
        print('requirements.txt file generated sucessfully.')
    except subprocess.CalledProcessError as e:
        print('error:', e)


export_requirements()

requirements.txt file generated sucessfully.


# 2. Dataset

In [9]:
df = pd.read_csv('data/dataset_final.csv')

df

,date,preco_cafe,cambio_brl,geada_Machado,geada_Manhuacu,geada_Patrocinio,geada_Varginha,precip_mm_sum_Machado,precip_mm_sum_Manhuacu,precip_mm_sum_Patrocinio,...,temp_max_C_max_Patrocinio,temp_max_C_max_Varginha,temp_min_C_min_Machado,temp_min_C_min_Manhuacu,temp_min_C_min_Patrocinio,temp_min_C_min_Varginha,umidade_pct_mean_Machado,umidade_pct_mean_Manhuacu,umidade_pct_mean_Patrocinio,umidade_pct_mean_Varginha
0,2019-01-02,99.500000,3.8799,0,0,0,0,0.0,0.0,0.2,...,28.1,30.100000,19.3,18.3,16.4,18.600000,82.208333,78.166667,84.333333,75.833333
1,2019-01-03,102.150002,3.7863,0,0,0,0,4.4,0.0,0.0,...,31.3,30.300000,19.4,19.1,16.9,19.800000,86.583333,69.875000,74.916667,80.208333
2,2019-01-04,101.599998,3.7551,0,0,0,0,60.4,3.6,4.2,...,25.8,25.400000,18.9,18.8,19.4,19.200000,92.875000,75.666667,83.375000,91.291667
3,2019-01-07,102.750000,3.6612,0,0,0,0,0.0,0.0,0.0,...,30.9,21.700000,19.2,19.3,15.6,18.500000,66.250000,76.958333,70.458333,87.600000
4,2019-01-08,105.050003,3.7341,0,0,0,0,0.0,0.0,0.2,...,32.1,21.886111,20.2,16.4,15.9,18.558333,65.375000,73.625000,75.458333,87.327381
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1757,2025-12-23,346.950012,5.5900,0,0,0,0,0.0,0.0,0.0,...,30.1,27.900000,20.1,18.8,15.0,26.500000,56.250000,75.608696,72.956522,78.000000
1758,2025-12-24,345.149994,5.5185,0,0,0,0,0.0,0.4,0.0,...,32.6,27.900000,21.2,19.6,16.1,26.500000,50.875000,77.000000,69.166667,78.000000
1759,2025-12-26,350.250000,5.5195,0,0,0,0,0.0,0.0,0.0,...,34.7,27.900000,23.7,17.3,15.9,26.500000,61.750000,62.947368,67.000000,78.000000
1760,2025-12-29,352.149994,5.5425,0,0,0,0,0.0,5.6,0.2,...,33.7,27.900000,21.9,19.0,17.4,26.500000,85.000000,70.958333,77.625000,78.000000


## 2.1. Data exploration